# CivilConflict

> 对应代码：`EconometricsCode/code_in_notes/Chap.17/CivilConflict.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.17`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.17")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先清空内存并关闭分页，随后加载 CivilConflict 数据集。该数据集来自 Miguel、Satyanath 和 Sergenti（2004）关于撒哈拉以南非洲国家内战与经济增长关系的经典研究。

In [ ]:
%%stata
clear
set more off
use ../datasets/CivilConflict

本例中，我们关心的内生解释变量是 GDP 增长率 gdp_g：经济增长可能通过降低冲突风险的内生渠道影响内战，也可能被内战反向影响（遗漏变量与反向因果并存）。工具变量是降水增长率 GPCP_g——在农业主导的非洲经济中，降水冲击是经济增长的外生来源，同时不直接影响冲突（满足排除性约束）。

In [ ]:
%%stata
// 内生变量：GDP增长率 gdp_g；工具变量：降水增长率 GPCP_g

我们先做第一阶段回归：把内生变量 gdp_g 对工具变量 GPCP_g 以及全部外生控制变量（国家固定效应 i.ccode、年份固定效应 i.year、国家特定线性时间趋势 i.ccode#c.year）做 OLS。这一步的目的是检查工具变量与内生变量的相关性——如果 GPCP_g 的系数显著，说明工具变量具有相关性；同时我们可以用 predict 得到拟合值 gdp_g_pred，即内生变量中由工具变量解释的"外生部分"。

In [ ]:
%%stata
// 第一阶段，first stage
reg gdp_g GPCP_g i.ccode i.year i.ccode#c.year
predict gdp_g_pred

接着我们估计约简式（reduced form）：直接把被解释变量 any_prio（是否发生内战）对工具变量 GPCP_g 及控制变量回归。由于工具变量是外生的，约简式给出的是降水对内战的总效应，它等于工具变量对内生变量的效应（第一阶段）乘以内生变量对被解释变量的效应。cl(ccode) 表示在国家层面聚类稳健标准误，以允许同一国家内不同年份的扰动项相关。

In [ ]:
%%stata
// reduced-form
reg any_prio GPCP_g i.ccode i.year i.ccode#c.year, cl(ccode)

这里演示一个常见的错误做法：手动把第一阶段得到的拟合值 gdp_g_pred 代入第二阶段做 OLS。虽然点估计与两阶段最小二乘（2SLS）一致，但标准误是错误的——它忽略了拟合值本身是估计出来的这一事实，没有计入第一阶段的不确定性，因此不能用于正确的统计推断。

In [ ]:
%%stata
// 错误做法：手动两阶段最小二乘
reg any_prio gdp_g_pred i.ccode i.year i.ccode#c.year,cl(ccode)

正确的做法是用专门的 2SLS 命令一步到位。ivregress 2sls 是 Stata 官方命令，括号内 (gdp_g = GPCP_g) 声明内生变量及其工具变量，命令自动完成两阶段估计并给出修正后的标准误；这里同样使用国家层面聚类稳健标准误。

In [ ]:
%%stata
// 两阶段最小二乘，三种命令
ivregress 2sls  any_prio (gdp_g = GPCP_g) i.ccode i.year i.ccode#c.year, cl(ccode)

ivreghdfe 是第三方命令，把 2SLS 与 reghdfe 的高维固定效应吸收能力结合起来：固定效应不必以虚拟变量形式进入回归，而是通过 absorb() 选项被"吸收"掉，在固定效应维度很高时大幅节省计算量。这里吸收国家、年份以及国家特定趋势三个维度的固定效应，结果与上一条命令完全等价。

In [ ]:
%%stata
ivreghdfe any_prio (gdp_g = GPCP_g), absorb(i.ccode  i.year i.ccode#c.year) cl(ccode)

ivreg2 是另一个广泛使用的第三方 2SLS 命令，语法与 ivregress 类似，但默认报告更丰富的诊断统计量（如弱工具变量检验的 Kleibergen-Paap 统计量、内生性检验等），是实证文献中最常用的 IV 估计命令之一。三条命令给出的系数估计相同，读者可以对比其输出。

In [ ]:
%%stata
ivreg2 any_prio (gdp_g = GPCP_g) i.ccode i.year i.ccode#c.year, cl(ccode)